## Karşılaştırılmak üzere tutulan SQL sorguları

### Veri tabanı bağlantısı

In [2]:
import sqlite3
import pandas as pd 

# Veritabanı dosyasının yolu
db_path = r"C:\Users\LENOVO\OneDrive\Masaüstü\Çalışmalar\Lotus AI\SQL Task\sqlite-sakila.db"  

# Bağlantı oluştur
conn = sqlite3.connect(db_path)

# Cursor oluştur
cursor = conn.cursor()

print("Veritabanına başarıyla bağlanıldı.")

Veritabanına başarıyla bağlanıldı.


## Tables:

### customer, category, film_category, language, actor, film_actor, film, staff, inventory, payment, rental, country, city, address, store


## SORULAR

1-) Tüm verileri göster

In [3]:
rental="SELECT * FROM rental"
rental_df = pd.read_sql_query(rental, conn)
print(rental_df)

       rental_id              rental_date  inventory_id  customer_id  \
0              1  2005-05-24 22:53:30.000           367          130   
1              2  2005-05-24 22:54:33.000          1525          459   
2              3  2005-05-24 23:03:39.000          1711          408   
3              4  2005-05-24 23:04:41.000          2452          333   
4              5  2005-05-24 23:05:21.000          2079          222   
...          ...                      ...           ...          ...   
16039      16045  2005-08-23 22:25:26.000           772           14   
16040      16046  2005-08-23 22:26:47.000          4364           74   
16041      16047  2005-08-23 22:42:48.000          2088          114   
16042      16048  2005-08-23 22:43:07.000          2019          103   
16043      16049  2005-08-23 22:50:12.000          2666          393   

                   return_date  staff_id          last_update  
0      2005-05-26 22:04:30.000         1  2021-03-06 15:53:41  
1      

2-) Her filmdeki oyuncuları listele

In [102]:
query= "SELECT a.first_name, a.last_name, f.title FROM actor a " \
"JOIN film_actor fa ON a.actor_id = fa.actor_id " \
"JOIN film f ON fa.film_id=f.film_id"

result=pd.read_sql_query(query, conn)

print(result)

     first_name last_name                  title
0      PENELOPE   GUINESS       ACADEMY DINOSAUR
1      PENELOPE   GUINESS   ANACONDA CONFESSIONS
2      PENELOPE   GUINESS            ANGELS LIFE
3      PENELOPE   GUINESS  BULWORTH COMMANDMENTS
4      PENELOPE   GUINESS          CHEAPER CLYDE
...         ...       ...                    ...
5457      THORA    TEMPLE       TELEGRAPH VOYAGE
5458      THORA    TEMPLE        TROJAN TOMORROW
5459      THORA    TEMPLE        VIRGINIAN PLUTO
5460      THORA    TEMPLE       WARDROBE PHANTOM
5461      THORA    TEMPLE         WRONG BEHAVIOR

[5462 rows x 3 columns]


3-) Her filmde kaç oyuncu oynadı?

In [5]:
query="SELECT f.title, COUNT(DISTINCT(fa.actor_id)) as actor_count FROM film f " \
"JOIN film_actor fa ON f.film_id=fa.film_id " \
"GROUP BY f.title " \
"ORDER BY actor_count DESC"
print(pd.read_sql_query(query, conn))

                title  actor_count
0    LAMBS CINCINATTI           15
1           RANDOM GO           13
2     MUMMY CREATURES           13
3     DRACULA CRYSTAL           13
4          CRAZY HOME           13
..                ...          ...
992     FERRIS MOTHER            1
993      DWARFS ALTER            1
994        DOLLS RAGE            1
995    BRIDE INTRIGUE            1
996   BAKED CLEOPATRA            1

[997 rows x 2 columns]


4-) Her oyuncu kaç filmde oynadı?

In [6]:
query="SELECT a.first_name,a.last_name,a.actor_id, COUNT(DISTINCT(fa.film_id)) as film_count FROM actor a " \
"JOIN film_actor fa ON a.actor_id=fa.actor_id " \
"GROUP BY a.actor_id " \
"ORDER BY film_count DESC"

print(pd.read_sql_query(query, conn))

    first_name  last_name  actor_id  film_count
0         GINA  DEGENERES       107          42
1       WALTER       TORN       102          41
2         MARY     KEITEL       198          40
3      MATTHEW     CARREY       181          39
4       SANDRA     KILMER        23          37
..         ...        ...       ...         ...
195       ADAM      GRANT        71          18
196      JULIA  ZELLWEGER       186          16
197       JUDY       DEAN        35          15
198      JULIA    FAWCETT       199          15
199      EMILY        DEE       148          14

[200 rows x 4 columns]


5-) Envanterde olmayan filmler var mı? Varsa kaç tane?

In [7]:
query="SELECT COUNT(DISTINCT(film_id)) as film_count FROM film_actor " \
"WHERE film_id NOT IN (SELECT DISTINCT(film_id) FROM inventory)"

print(pd.read_sql_query(query, conn))

   film_count
0          42


6-) Kiralanabilir olan her filmin kaç kez kiralandığını ve toplam gelirlerini getirin

In [8]:
query="SELECT f.title, COUNT(DISTINCT(r.rental_id)) as rental_count, SUM(p.amount) as total_revenue FROM rental r " \
"JOIN inventory i ON r.inventory_id=i.inventory_id " \
"JOIN film f ON i.film_id=f.film_id " \
"JOIN payment p ON r.rental_id=p.rental_id " \
"GROUP BY f.title " \
"ORDER BY rental_count DESC"

print(pd.read_sql_query(query, conn))

                   title  rental_count  total_revenue
0     BUCKET BROTHERHOOD            34         180.66
1       ROCKETEER MOTHER            33         116.67
2          SCALAWAG DUCK            32         172.68
3    RIDGEMONT SUBMARINE            32         130.68
4         JUGGLER HARDLY            32          96.68
..                   ...           ...            ...
953          BUNCH MINDS             5          25.95
954     BRAVEHEART HUMAN             5          16.95
955          TRAIN BUNCH             4          24.96
956          MIXED DOORS             4          15.96
957       HARDLY ROBBERS             4          15.96

[958 rows x 3 columns]


7-) Envanterde olmayan filmlerin kira oranlarını getirin

In [9]:
query="SELECT title, rental_rate FROM film " \
" WHERE film_id NOT IN (SELECT DISTINCT(film_id) FROM inventory)"
print(pd.read_sql_query(query,conn))

                     title  rental_rate
0           ALICE FANTASIA         0.99
1              APOLLO TEEN         2.99
2           ARGONAUTS TOWN         0.99
3            ARK RIDGEMONT         0.99
4     ARSENIC INDEPENDENCE         0.99
5        BOONDOCK BALLROOM         0.99
6            BUTCH PANTHER         0.99
7            CATCH AMISTAD         0.99
8      CHINATOWN GLADIATOR         4.99
9           CHOCOLATE DUCK         2.99
10    COMMANDMENTS EXPRESS         4.99
11        CROSSING DIVORCE         4.99
12         CROWDS TELEMARK         4.99
13        CRYSTAL BREAKING         2.99
14              DAZED PUNK         4.99
15  DELIVERANCE MULHOLLAND         0.99
16       FIREHOUSE VIETNAM         0.99
17           FLOATS GARDEN         2.99
18   FRANKENSTEIN STRANGER         0.99
19      GLADIATOR WESTWARD         4.99
20               GUMP DATE         4.99
21           HATE HANDICAP         0.99
22             HOCUS FRIDA         2.99
23        KENTUCKIAN GIANT         2.99


8-) Birden fazla DVD'yi iade etmeyen kaç müşteri var?

In [10]:
query="SELECT COUNT(*) AS customer_count FROM " \
" (SELECT customer_id FROM rental " \
" WHERE return_date IS NULL " \
" GROUP BY customer_id " \
" HAVING COUNT(rental_id) > 1 " \
" )"

print(pd.read_sql_query(query,conn))

   customer_count
0              23


9-) Her müşteri kaç film kiraladı?

In [11]:
query=" SELECT c.customer_id,c.first_name, c.last_name, COUNT(r.rental_id) AS rental_count FROM rental r " \
"JOIN customer c ON r.customer_id=c.customer_id " \
"GROUP BY c.customer_id, c.first_name, c.last_name " \
"ORDER BY rental_count DESC " \

print(pd.read_sql_query(query,conn))

     customer_id first_name last_name  rental_count
0            148    ELEANOR      HUNT            46
1            526       KARL      SEAL            45
2            144      CLARA      SHAW            42
3            236     MARCIA      DEAN            42
4             75      TAMMY   SANDERS            41
..           ...        ...       ...           ...
594          248   CAROLINE    BOWMAN            15
595           61  KATHERINE    RIVERA            14
596          110    TIFFANY    JORDAN            14
597          281      LEONA    OBRIEN            14
598          318      BRIAN     WYMAN            12

[599 rows x 4 columns]


10-) Türlerine göre en çok kiralanan filmler ve bunlara ne kadar ödendi?

In [104]:
query = """
SELECT category.name AS genre, film.title, COUNT(rental.rental_id) AS rental_count, SUM(payment.amount) AS total_revenue
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY category.name, film.title
ORDER BY rental_count DESC, title ASC;
"""

df = pd.read_sql_query(query, conn)
print(df)

           genre               title  rental_count  total_revenue
0         Travel  BUCKET BROTHERHOOD            34         180.66
1        Foreign    ROCKETEER MOTHER            33         116.67
2          Games      FORWARD TEMPLE            32         128.68
3          Games      GRIT CLOCKWORK            32         110.68
4      Animation      JUGGLER HARDLY            32          96.68
..           ...                 ...           ...            ...
953        Games         SEVEN SWARM             5          30.95
954       Travel      TRAFFIC HOBBIT             5          24.95
955  Documentary      HARDLY ROBBERS             4          15.96
956      Foreign         MIXED DOORS             4          15.96
957       Horror         TRAIN BUNCH             4          24.96

[958 rows x 4 columns]


11-) Tür ve tarihe göre kiralama sayısı ve gelir

In [13]:
query = """
SELECT
    category.name AS genre,
    DATE(rental.rental_date) AS rental_date,
    COUNT(rental.rental_id) AS rental_count,
    SUM(payment.amount) AS total_revenue
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY category.name, DATE(rental.rental_date)
ORDER BY rental_date DESC;
"""

df = pd.read_sql_query(query, conn)
print(df)

         genre rental_date  rental_count  total_revenue
0       Action  2006-02-14            17          55.80
1    Animation  2006-02-14            21          66.75
2     Children  2006-02-14             6          15.94
3     Classics  2006-02-14             9          24.89
4       Comedy  2006-02-14             9          25.90
..         ...         ...           ...            ...
628   Children  2005-05-24             2          10.98
629     Comedy  2005-05-24             1           0.99
630     Family  2005-05-24             1           2.99
631     Horror  2005-05-24             2           6.98
632      Music  2005-05-24             1           2.99

[633 rows x 4 columns]


12-) Kiralanabilir filmler için türlerine göre her filmin kaç kez kiralandığı

In [14]:
query="""
SELECT c.name ,f.title, COUNT(r.rental_id) AS rental_count FROM film f
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
GROUP BY c.name, f.title
"""

print(pd.read_sql_query(query, conn))

       name                title  rental_count
0    Action         AMADEUS HOLY            21
1    Action      AMERICAN CIRCUS            22
2    Action   ANTITRUST TOMATOES            10
3    Action  BAREFOOT MANCHURIAN            18
4    Action         BERETS AGENT            21
..      ...                  ...           ...
953  Travel  VALENTINE VANISHING            12
954  Travel          WINDOW SIDE            12
955  Travel        WOLVES DESIRE            21
956  Travel        WORKER TARZAN            15
957  Travel  WORKING MICROCOSMOS            25

[958 rows x 3 columns]


13-) En çok rafta bekleyen filmler

In [15]:
query="""
SELECT film.title,
       CAST(julianday('now') - julianday(rental.rental_date) AS INTEGER) AS days_on_shelf
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE rental.return_date IS NULL
ORDER BY days_on_shelf DESC;
"""
print(pd.read_sql_query(query, conn))

                     title  days_on_shelf
0         ACADEMY DINOSAUR           7624
1              HYDE DOCTOR           7446
2              HUNGER ROOF           7446
3           FRISCO FORREST           7446
4              TITANS JERK           7446
..                     ...            ...
178            DANCES NONE           7446
179  CONVERSATION DOWNHILL           7446
180            SHOCK CABIN           7446
181         WEDDING APOLLO           7446
182            WINDOW SIDE           7446

[183 rows x 2 columns]


SQlite'da MySQL deki gibi DATEDIFF ve NOW yokmuş dolayısıyla bu tarz bir çözüm önerildi.

14-) Geç, erken ve zamanında iade edilmiş kiralanmış filmler

In [16]:
query = """
SELECT
    SUM(CASE WHEN date(rental.return_date) > date(rental.rental_date, '+' || film.rental_duration || ' days') THEN 1 ELSE 0 END) AS late_returns,
    SUM(CASE WHEN date(rental.return_date) < date(rental.rental_date, '+' || film.rental_duration || ' days') THEN 1 ELSE 0 END) AS early_returns,
    SUM(CASE WHEN date(rental.return_date) = date(rental.rental_date, '+' || film.rental_duration || ' days') THEN 1 ELSE 0 END) AS on_time_returns
FROM rental
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
WHERE rental.return_date IS NOT NULL;
"""
print(pd.read_sql_query(query, conn))

   late_returns  early_returns  on_time_returns
0          7269           6864             1728


15-) Hangi müşteri en çok DVD kiralamış?

In [101]:
query="""
SELECT c.customer_id,c.first_name, c.last_name, (COUNT(r.rental_id)) as total_rentals FROM rental r
JOIN customer c ON r.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
ORDER BY total_rentals DESC
LIMIT 1
"""
print(pd.read_sql_query(query, conn))

   customer_id first_name last_name  total_rentals
0          148    ELEANOR      HUNT             46


16-)En popüler film kategorisi nedir?

In [18]:
query = """
SELECT category.name, COUNT(rental.rental_id) AS rental_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY category.name
ORDER BY rental_count DESC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

     name  rental_count
0  Sports          1179


17-) Hangi çalışan en çok kiralama işlemi gerçekleştirmiş?

In [19]:
query="""
SELECT s.first_name, s.last_name, COUNT(r.rental_id) AS total_rental
FROM rental r
JOIN staff s ON r.staff_id = s.staff_id
GROUP BY s.staff_id, s.first_name, s.last_name
LIMIT 1
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  total_rental
0       Mike   Hillyer          8040


18-) En çok geliri hangi film getirmiş?

In [20]:
query="""
SELECT f.title, SUM(p.amount) AS total_revenue
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
JOIN payment p ON r.rental_id = p.rental_id
GROUP BY f.title
ORDER BY total_revenue DESC
LIMIT 1

"""
print(pd.read_sql_query(query,conn))

              title  total_revenue
0  TELEGRAPH VOYAGE         231.73


19-) Her müşteri için toplam harcama miktarı

In [21]:
query="""
SELECT c.first_name, c.last_name, SUM(p.amount) AS total_spent
FROM customer c
JOIN payment p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
ORDER BY total_spent DESC
"""
print(pd.read_sql_query(query, conn))

    first_name last_name  total_spent
0         KARL      SEAL       221.55
1      ELEANOR      HUNT       216.54
2        CLARA      SHAW       195.58
3       RHONDA   KENNEDY       194.61
4       MARION    SNYDER       194.61
..         ...       ...          ...
594      ANNIE   RUSSELL        58.82
595     JOHNNY    TURPIN        57.81
596      BRIAN     WYMAN        52.88
597      LEONA    OBRIEN        50.86
598   CAROLINE    BOWMAN        50.85

[599 rows x 3 columns]


20-) Her kategorideki toplam kiralama sayısını ve gelirleri bululn

In [22]:
query="""
SELECT c.name, COUNT(r.rental_id) AS rental_count, SUM(p.amount) AS total_revenue
FROM category c
JOIN film_category fc ON c.category_id = fc.category_id
JOIN film f ON fc.film_id = f.film_id
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
JOIN payment p ON r.rental_id = p.rental_id
GROUP BY c.name
ORDER BY rental_count DESC
"""
print(pd.read_sql_query(query,conn))

           name  rental_count  total_revenue
0        Sports          1179        5314.21
1     Animation          1166        4656.30
2        Action          1112        4375.85
3        Sci-Fi          1101        4756.98
4        Family          1096        4226.07
5         Drama          1060        4587.39
6   Documentary          1050        4217.52
7       Foreign          1033        4270.67
8         Games           969        4281.33
9      Children           945        3655.55
10       Comedy           941        4383.58
11          New           940        4351.62
12     Classics           939        3639.59
13       Horror           846        3722.54
14       Travel           837        3549.64
15        Music           830        3417.72


21-) En uzun süre kirada kalmış filmleri bulun

In [23]:
query = """
SELECT f.title,
       julianday(r.return_date) - julianday(r.rental_date) AS duration
FROM inventory i
JOIN film f ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
WHERE r.return_date IS NOT NULL
ORDER BY duration DESC;
"""
print(pd.read_sql_query(query, conn))

                    title  duration
0      HOLOCAUST HIGHBALL  9.249306
1         HIGHBALL POTTER  9.249306
2              PANIC CLUB  9.248611
3       CONEHEADS SMOOCHY  9.248611
4            TRAMP OTHERS  9.248611
...                   ...       ...
15856       KISSING DOLLS  0.751389
15857      HEAVEN FREEDOM  0.751389
15858        ROAD ROXANNE  0.750694
15859   CURTAIN VIDEOTAPE  0.750000
15860     DISCIPLE MOTHER  0.750000

[15861 rows x 2 columns]


22-) En az kiralanan 5 film hangileridir?

In [105]:
query="""
SELECT f.title, COUNT(r.rental_id) as count FROM film f
JOIN inventory i ON i.film_id=f.film_id
JOIN rental r ON r.inventory_id= i.inventory_id
GROUP BY (f.title)
ORDER BY count ASC, f.title ASC
LIMIT 5

"""
print(pd.read_sql_query(query,conn))

              title  count
0    HARDLY ROBBERS      4
1       MIXED DOORS      4
2       TRAIN BUNCH      4
3  BRAVEHEART HUMAN      5
4       BUNCH MINDS      5


24-) En fazla kazanç sağlayan 5 müşteriyi bulun

In [25]:
query="""
SELECT c.customer_id, c.first_name,c.last_name, SUM(p.amount) AS total FROM customer c
JOIN payment p ON p.customer_id=c.customer_id
GROUP BY c.customer_id, c.first_name,c.last_name
ORDER BY total DESC
LIMIT 5

"""
print(pd.read_sql_query(query,conn))

   customer_id first_name last_name   total
0          526       KARL      SEAL  221.55
1          148    ELEANOR      HUNT  216.54
2          144      CLARA      SHAW  195.58
3          137     RHONDA   KENNEDY  194.61
4          178     MARION    SNYDER  194.61


25-) Her filmin ortalama kiralanma süresini bulun

In [26]:
query = """
SELECT film.title, AVG(julianday(rental.return_date) - julianday(rental.rental_date)) AS avg_rental_duration
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE rental.return_date IS NOT NULL
GROUP BY film.title;
"""
print(pd.read_sql_query(query, conn))

                 title  avg_rental_duration
0     ACADEMY DINOSAUR             4.997128
1       ACE GOLDFINGER             5.642708
2     ADAPTATION HOLES             3.465625
3     AFFAIR PREJUDICE             4.758333
4          AFRICAN EGG             7.106629
..                 ...                  ...
953     YOUNG LANGUAGE             4.624405
954         YOUTH KICK             5.562963
955       ZHIVAGO CORE             5.950087
956  ZOOLANDER FICTION             5.542647
957          ZORRO ARK             4.539180

[958 rows x 2 columns]


26-) Her türde en popüler filmi bulun

In [27]:
query="""
SELECT c.name, f.title, COUNT(r.rental_id) AS count FROM film f
JOIN film_category fc ON fc.film_id=f.film_id
JOIN category c ON c.category_id=fc.category_id
JOIN inventory i ON i.film_id=f.film_id
JOIN rental r ON r.inventory_id=i.inventory_id
GROUP BY c.name
ORDER BY count DESC

"""
print(pd.read_sql_query(query,conn))

           name                title  count
0        Sports      PIRATES ROXANNE   1179
1     Animation       LAWLESS VISION   1166
2        Action       EASY GLADIATOR   1112
3        Sci-Fi      HOLLOW JEOPARDY   1101
4        Family      BLANKET BEVERLY   1096
5         Drama         HANGING DEEP   1060
6   Documentary     PELICAN COMFORTS   1050
7       Foreign       MATRIX SNOWMAN   1033
8         Games         WIND PHANTOM    969
9      Children        GRADUATE LORD    945
10       Comedy        MYSTIC TRUMAN    941
11          New  RIDGEMONT SUBMARINE    940
12     Classics         HOPE TOOTSIE    939
13       Horror        LOVE SUICIDES    846
14       Travel      GAMES BOWFINGER    837
15        Music         FREAKY POCUS    830


27-) Her türde en fazla gelir sağlayan filmi bulun

In [28]:
query="""
SELECT c.name, f.title, SUM(p.amount) AS revenue FROM film f
JOIN film_category fc ON fc.film_id=f.film_id
JOIN category c ON c.category_id=fc.category_id
JOIN inventory i ON i.film_id=f.film_id
JOIN rental r ON r.inventory_id=i.inventory_id
JOIN payment p ON p.rental_id=r.rental_id
GROUP BY c.name
ORDER BY revenue DESC
"""
print(pd.read_sql_query(query,conn))

           name                title  revenue
0        Sports    TALENTED HOMICIDE  5314.21
1        Sci-Fi         ATTACKS HATE  4756.98
2     Animation      DOORS PRESIDENT  4656.30
3         Drama        SAVANNAH TOWN  4587.39
4        Comedy        FERRIS MOTHER  4383.58
5        Action         WOMEN DORADO  4375.85
6           New    AMISTAD MIDSUMMER  4351.62
7         Games          FIRE WOLVES  4281.33
8       Foreign   USUAL UNTOUCHABLES  4270.67
9        Family     FINDING ANACONDA  4226.07
10  Documentary     ADAPTATION HOLES  4217.52
11       Horror   STRANGERS GRAFFITI  3722.54
12     Children    ZOOLANDER FICTION  3655.55
13     Classics       PATIENT SISTER  3639.59
14       Travel  EXPECATIONS NATURAL  3549.64
15        Music           YOUTH KICK  3417.72


28-) En çok DVD iade etmeyen müşteriyi bulun

In [29]:
query = """
SELECT customer.first_name, customer.last_name, COUNT(rental.rental_id) AS unreturned_count
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
WHERE rental.return_date IS NULL
GROUP BY customer.first_name, customer.last_name
ORDER BY unreturned_count DESC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  unreturned_count
0      TAMMY   SANDERS                 3


29-) En fazla kiralama yapan 5 çalışanı bulun

In [30]:
query = """
SELECT staff.first_name, staff.last_name, COUNT(rental.rental_id) AS rental_count
FROM staff
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY staff.first_name, staff.last_name
ORDER BY rental_count DESC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  rental_count
0       Mike   Hillyer          8040
1        Jon  Stephens          8004


30-) En fazla kiralama yapan 5 müşteri hangi şubeden kiralama yapmıştır

In [31]:
query = """
SELECT store.store_id, customer.first_name, customer.last_name, COUNT(rental.rental_id) AS rental_count
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN store ON inventory.store_id = store.store_id
GROUP BY store.store_id, customer.first_name, customer.last_name
ORDER BY rental_count DESC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

   store_id first_name last_name  rental_count
0         2      JORGE  OLIVARES            26
1         1   GERTRUDE  CASTILLO            25
2         1     MARCIA      DEAN            25
3         2      CLARA      SHAW            25
4         2    ELEANOR      HUNT            25


31-) Her türde en az kiralanan filmi bulun

In [ ]:
query = """
SELECT category.name AS genre, film.title, COUNT(rental.rental_id) AS rental_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY genre
ORDER BY rental_count ASC
"""
print(pd.read_sql_query(query, conn))

          genre                title  rental_count
0         Music         FREAKY POCUS           830
1        Travel      GAMES BOWFINGER           837
2        Horror        LOVE SUICIDES           846
3      Classics         HOPE TOOTSIE           939
4           New  RIDGEMONT SUBMARINE           940
5        Comedy        MYSTIC TRUMAN           941
6      Children        GRADUATE LORD           945
7         Games         WIND PHANTOM           969
8       Foreign       MATRIX SNOWMAN          1033
9   Documentary     PELICAN COMFORTS          1050
10        Drama         HANGING DEEP          1060
11       Family      BLANKET BEVERLY          1096
12       Sci-Fi      HOLLOW JEOPARDY          1101
13       Action       EASY GLADIATOR          1112
14    Animation       LAWLESS VISION          1166
15       Sports      PIRATES ROXANNE          1179


32-) En çok kiralama yapan 5 müşteri hangi şehirde?

In [33]:
query = """
SELECT city.city, customer.first_name, customer.last_name, COUNT(rental.rental_id) AS rental_count
FROM customer
JOIN address ON customer.address_id = address.address_id
JOIN city ON address.city_id = city.city_id
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY city.city, customer.first_name, customer.last_name
ORDER BY rental_count DESC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

          city first_name last_name  rental_count
0  Saint-Denis    ELEANOR      HUNT            46
1   Cape Coral       KARL      SEAL            45
2    Molodetno      CLARA      SHAW            42
3        Tanza     MARCIA      DEAN            42
4     Changhwa      TAMMY   SANDERS            41


33-)En çok kazanç sağlayan 5 müşteri hangi şehirde

In [34]:
query = """
SELECT city.city, customer.first_name, customer.last_name, SUM(payment.amount) AS total_spent
FROM customer
JOIN address ON customer.address_id = address.address_id
JOIN city ON address.city_id = city.city_id
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY city.city, customer.first_name, customer.last_name
ORDER BY total_spent DESC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

                  city first_name last_name  total_spent
0           Cape Coral       KARL      SEAL       221.55
1          Saint-Denis    ELEANOR      HUNT       216.54
2            Molodetno      CLARA      SHAW       195.58
3            Apeldoorn     RHONDA   KENNEDY       194.61
4  Santa Brbara dOeste     MARION    SNYDER       194.61


34-) En çok kiralanan 5 film hangi şehirde

In [106]:
query = """
SELECT city.city, film.title, COUNT(rental.rental_id) AS rental_count
FROM city
JOIN address ON city.city_id = address.city_id
JOIN customer ON address.address_id = customer.address_id
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
GROUP BY city.city, film.title
ORDER BY rental_count DESC, film.title ASC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

       city              title  rental_count
0  Sorocaba    CADDYSHACK JEDI             3
1    Kurgan   DETECTIVE VISION             3
2      Lima    DISCIPLE MOTHER             3
3   Trshavn  FLATLINERS KILLER             3
4   Caracas   AFFAIR PREJUDICE             2


35-) En az kiralanan 5 film hangi şehirde

In [36]:
query = """
SELECT city.city, film.title, COUNT(rental.rental_id) AS rental_count
FROM city
JOIN address ON city.city_id = address.city_id
JOIN customer ON address.address_id = customer.address_id
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
GROUP BY city.city, film.title
ORDER BY rental_count ASC, film.title ASC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

               city             title  rental_count
0   Almirante Brown  ACADEMY DINOSAUR             1
1            Celaya  ACADEMY DINOSAUR             1
2  Charlotte Amalie  ACADEMY DINOSAUR             1
3           Cianjur  ACADEMY DINOSAUR             1
4            Goinia  ACADEMY DINOSAUR             1


36-) En çok kazanç sağlayan 5 film hangi şehirde

In [37]:
query = """
SELECT city.city, film.title, SUM(payment.amount) AS total_revenue
FROM city
JOIN address ON city.city_id = address.city_id
JOIN customer ON address.address_id = customer.address_id
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY city.city, film.title
ORDER BY total_revenue DESC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

                  city             title  total_revenue
0                Plock  CALIFORNIA BIRDS          18.98
1          Probolinggo      MINDS TRUMAN          18.98
2               Kamyin    ROSES TREASURE          17.98
3  Santa Brbara dOeste         WIFE TURN          17.98
4              Bijapur   WHISPERER GIANT          16.98


37-) En az kazanç sağlayan 5 film hangi şehirde

In [38]:
query = """
SELECT city.city, film.title, SUM(payment.amount) AS total_revenue
FROM city
JOIN address ON city.city_id = address.city_id
JOIN customer ON address.address_id = customer.address_id
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY city.city, film.title
ORDER BY total_revenue ASC, title ASC
LIMIT 5;
"""
print(pd.read_sql_query(query, conn))

        city              title  total_revenue
0      Merlo       BLADE POLISH              0
1     Jaipur    CHAMBER ITALIAN              0
2       Bern      CHASING FIGHT              0
3   Rockford    CLEOPATRA DEVIL              0
4  Balurghat  CURTAIN VIDEOTAPE              0


38-)En fazla kiralama yapan müşteri hangi filmleri kiralamış

In [39]:
query = """
SELECT customer.first_name, customer.last_name, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
WHERE customer.customer_id = (
    SELECT customer.customer_id
    FROM customer
    JOIN rental ON customer.customer_id = rental.customer_id
    GROUP BY customer.customer_id
    ORDER BY COUNT(rental.rental_id) DESC
    LIMIT 1
)
GROUP BY customer.first_name, customer.last_name, genre;
"""
print(pd.read_sql_query(query, conn))

   first_name last_name        genre  rental_count
0     ELEANOR      HUNT       Action             3
1     ELEANOR      HUNT     Classics             4
2     ELEANOR      HUNT       Comedy             2
3     ELEANOR      HUNT  Documentary             3
4     ELEANOR      HUNT        Drama             1
5     ELEANOR      HUNT       Family             6
6     ELEANOR      HUNT      Foreign             4
7     ELEANOR      HUNT        Games             3
8     ELEANOR      HUNT       Horror             4
9     ELEANOR      HUNT        Music             1
10    ELEANOR      HUNT          New             2
11    ELEANOR      HUNT       Sci-Fi             7
12    ELEANOR      HUNT       Sports             1
13    ELEANOR      HUNT       Travel             5


40-)En çok kazanç sağlayan müşteri hangi filmleri kiralamış

In [40]:
query = """
SELECT customer.first_name, customer.last_name, film.title, SUM(payment.amount) AS total_spent
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE customer.customer_id = (
    SELECT customer.customer_id
    FROM customer
    JOIN rental ON customer.customer_id = rental.customer_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY customer.customer_id
    ORDER BY SUM(payment.amount) DESC
    LIMIT 1
)
GROUP BY customer.first_name, customer.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

   first_name last_name                  title  total_spent
0        KARL      SEAL          BETRAYED REAR         4.99
1        KARL      SEAL       BIKINI BORROWERS         6.99
2        KARL      SEAL         BLUES INSTINCT         2.99
3        KARL      SEAL    BRINGING HYSTERICAL         2.99
4        KARL      SEAL         CYCLONE FAMILY         4.99
5        KARL      SEAL             DATE SPEED         3.99
6        KARL      SEAL           DESIRE ALIEN         2.99
7        KARL      SEAL       DESTINY SATURDAY         4.99
8        KARL      SEAL             DUMBO LUST         0.99
9        KARL      SEAL           DURHAM PANKY         4.99
10       KARL      SEAL       ENGLISH BULWORTH         5.99
11       KARL      SEAL         FIDELITY DEVIL         4.99
12       KARL      SEAL       FOOL MOCKINGBIRD        10.99
13       KARL      SEAL  FORRESTER COMANCHEROS         4.99
14       KARL      SEAL            GANGS PRIDE         2.99
15       KARL      SEAL        GOLDMINE 

41-) En az kazanç sağlayan müşteri hangi filmleri kiralamış

In [41]:
query = """
SELECT customer.first_name, customer.last_name, film.title, SUM(payment.amount) AS total_spent
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE customer.customer_id = (
    SELECT customer.customer_id
    FROM customer
    JOIN rental ON customer.customer_id = rental.customer_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY customer.customer_id
    ORDER BY SUM(payment.amount) ASC
    LIMIT 1
)
GROUP BY customer.first_name, customer.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

   first_name last_name                   title  total_spent
0    CAROLINE    BOWMAN         ARMAGEDDON LOST         0.99
1    CAROLINE    BOWMAN        CAMELOT VACATION         0.99
2    CAROLINE    BOWMAN         DISCIPLE MOTHER         5.99
3    CAROLINE    BOWMAN        EMPIRE MALKOVICH         0.99
4    CAROLINE    BOWMAN          FRENCH HOLIDAY         7.99
5    CAROLINE    BOWMAN       GOODFELLAS SALUTE         4.99
6    CAROLINE    BOWMAN       HOOSIERS BIRDCAGE         4.99
7    CAROLINE    BOWMAN         ILLUSION AMELIE         0.99
8    CAROLINE    BOWMAN    IMPOSSIBLE PREJUDICE         5.99
9    CAROLINE    BOWMAN              JADE BUNCH         2.99
10   CAROLINE    BOWMAN         KILLER INNOCENT         2.99
11   CAROLINE    BOWMAN              MASK PEACH         3.99
12   CAROLINE    BOWMAN  RESURRECTION SILVERADO         0.99
13   CAROLINE    BOWMAN             TEXAS WATCH         0.99
14   CAROLINE    BOWMAN         WHISPERER GIANT         4.99


42-) En az kazanç sağlayan müşteri hangi türde en fazla film kiralamış

In [42]:
query = """
SELECT customer.first_name, customer.last_name, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
WHERE customer.customer_id = (
    SELECT customer.customer_id
    FROM customer
    JOIN rental ON customer.customer_id = rental.customer_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY customer.customer_id
    ORDER BY SUM(payment.amount) ASC
    LIMIT 1
)
GROUP BY customer.first_name, customer.last_name, genre
ORDER BY rental_count DESC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name   genre  rental_count
0   CAROLINE    BOWMAN  Sci-Fi             4


43-) En çok kiralanan film hangi çalışan tarafından kiralanmış

In [43]:
query = """
SELECT staff.first_name, staff.last_name, film.title, COUNT(rental.rental_id) AS rental_count
FROM staff
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY film.film_id
    ORDER BY COUNT(rental.rental_id) DESC
    LIMIT 1
)
GROUP BY staff.first_name, staff.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name               title  rental_count
0        Jon  Stephens  BUCKET BROTHERHOOD            18
1       Mike   Hillyer  BUCKET BROTHERHOOD            16


44-) En az kiralanan film hangi çalışan tarafından kiralanmış

In [103]:
query = """
SELECT staff.first_name, staff.last_name, film.title, COUNT(rental.rental_id) AS rental_count
FROM staff
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY film.film_id
    ORDER BY COUNT(rental.rental_id) ASC, title ASC
    LIMIT 1
)
GROUP BY staff.first_name, staff.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name           title  rental_count
0        Jon  Stephens  HARDLY ROBBERS             3
1       Mike   Hillyer  HARDLY ROBBERS             1


45-) En çok kazanç sağlayan film hangi çalışan tarafından kiralanmış

In [45]:
query = """
SELECT staff.first_name, staff.last_name, film.title, SUM(payment.amount) AS total_revenue
FROM staff
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id
    ORDER BY SUM(payment.amount) DESC
    LIMIT 1
)
GROUP BY staff.first_name, staff.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name             title  total_revenue
0        Jon  Stephens  TELEGRAPH VOYAGE         147.83
1       Mike   Hillyer  TELEGRAPH VOYAGE          83.90


46-)En az kazanç sağlayan film hangi çalışan tarafından kiralanmış


In [46]:
query = """
SELECT staff.first_name, staff.last_name, film.title, SUM(payment.amount) AS total_revenue
FROM staff
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id
    ORDER BY SUM(payment.amount) ASC
    LIMIT 1
)
GROUP BY staff.first_name, staff.last_name, film.title;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name             title  total_revenue
0        Jon  Stephens  OKLAHOMA JUMANJI           1.98
1       Mike   Hillyer  OKLAHOMA JUMANJI           3.96


47-)En çok kiralanan film hangi mağazada kiralanmış

In [47]:
query = """
SELECT store.store_id, film.title, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY film.film_id
    ORDER BY COUNT(rental.rental_id) DESC
    LIMIT 1
)
GROUP BY store.store_id, film.title;
"""
print(pd.read_sql_query(query, conn))

   store_id               title  rental_count
0         1  BUCKET BROTHERHOOD            16
1         2  BUCKET BROTHERHOOD            18


48-) En az kiralanan film hangi mağazada kiralanmış

In [48]:
query = """
SELECT store.store_id, film.title, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY film.film_id
    ORDER BY COUNT(rental.rental_id) ASC
    LIMIT 1
)
GROUP BY store.store_id, film.title;
"""
print(pd.read_sql_query(query, conn))

   store_id           title  rental_count
0         1  HARDLY ROBBERS             1
1         2  HARDLY ROBBERS             3


49-) En çok kazanç sağlayan film hangi mağazada kiralanmış

In [49]:
query = """
SELECT store.store_id, film.title, SUM(payment.amount) AS total_revenue
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id
    ORDER BY SUM(payment.amount) DESC
    LIMIT 1
)
GROUP BY store.store_id, film.title;
"""
print(pd.read_sql_query(query, conn))

   store_id             title  total_revenue
0         1  TELEGRAPH VOYAGE          83.90
1         2  TELEGRAPH VOYAGE         147.83


50-) En az kazanç sağlayan film hangi mağazada kiralanmış

In [50]:
query = """
SELECT store.store_id, film.title, SUM(payment.amount) AS total_revenue
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN payment ON rental.rental_id = payment.rental_id
WHERE film.film_id = (
    SELECT film.film_id
    FROM film
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id
    ORDER BY SUM(payment.amount) ASC
    LIMIT 1
)
GROUP BY store.store_id, film.title;
"""
print(pd.read_sql_query(query, conn))

   store_id             title  total_revenue
0         1  OKLAHOMA JUMANJI           3.96
1         2  OKLAHOMA JUMANJI           1.98


51-) Müşterilerin kiraladıkları filmlerin toplam kiralama süresi ne kadardır?

In [51]:
query = """
SELECT customer.first_name, customer.last_name,
       SUM(julianday(rental.return_date) - julianday(rental.rental_date)) AS total_rental_days
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY customer.first_name, customer.last_name;
"""
print(pd.read_sql_query(query, conn))

    first_name last_name  total_rental_days
0        AARON     SELBY         120.398611
1         ADAM     GOOCH         116.220833
2       ADRIAN     CLARY         101.916667
3        AGNES    BISHOP         111.152778
4         ALAN      KAHN         143.023611
..         ...       ...                ...
594     WILLIE   MARKHAM         125.833333
595      WILMA  RICHARDS         117.254861
596    YOLANDA    WEAVER         139.776389
597     YVONNE   WATKINS          97.145139
598    ZACHARY      HITE         169.749306

[599 rows x 3 columns]


52-) En çok kiralanan türdeki filmler hangileridir?

In [52]:
query = """
SELECT film.title, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM film
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE category.name = (
    SELECT category.name
    FROM category
    JOIN film_category ON category.category_id = film_category.category_id
    JOIN film ON film_category.film_id = film.film_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY category.name
    ORDER BY COUNT(rental.rental_id) DESC
    LIMIT 1
)
GROUP BY film.title, genre
ORDER BY rental_count DESC, title DESC;
"""
print(pd.read_sql_query(query, conn))

                  title   genre  rental_count
0     TALENTED HOMICIDE  Sports            29
1   GLEAMING JAWBREAKER  Sports            29
2        SATURDAY LAMBS  Sports            28
3        ROSES TREASURE  Sports            28
4   CHANCE RESURRECTION  Sports            27
..                  ...     ...           ...
68     LESSON CLEOPATRA  Sports             7
69      DIVORCE SHINING  Sports             7
70   GRACELAND DYNAMITE  Sports             6
71        BUBBLE GROSSE  Sports             6
72   MUSSOLINI SPOILERS  Sports             5

[73 rows x 3 columns]


53-) En az kiralanan türdeki filmler hangileridir?

In [53]:
query = """
SELECT film.title, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM film
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE category.name = (
    SELECT category.name
    FROM category
    JOIN film_category ON category.category_id = film_category.category_id
    JOIN film ON film_category.film_id = film.film_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY category.name
    ORDER BY COUNT(rental.rental_id) ASC
    LIMIT 1
)
GROUP BY film.title, genre
ORDER BY rental_count ASC;
"""
print(pd.read_sql_query(query, conn))

                     title  genre  rental_count
0          TERMINATOR CLUB  Music             6
1               YOUTH KICK  Music             6
2               CLUE GRAIL  Music             7
3               ELF MURDER  Music             7
4             HEAVENLY GUN  Music             7
5         GROSSE WONDERFUL  Music             8
6            REBEL AIRPORT  Music             8
7           RUNNER MADIGAN  Music             8
8          BIRDCAGE CASPER  Music             9
9           IMPACT ALADDIN  Music             9
10             LEGEND JEDI  Music             9
11              REDS POCUS  Music             9
12               WON DARES  Music             9
13     AMELIE HELLFIGHTERS  Music            10
14     JAWBREAKER BROOKLYN  Music            10
15           OLEANDER CLUE  Music            10
16         BIRCH ANTITRUST  Music            11
17             SONG HEDWIG  Music            11
18           MINORITY KISS  Music            12
19         CHAMBER ITALIAN  Music       

54-) Her müşteri için toplam ödeme miktarını bulun

In [54]:
query = """
SELECT customer.first_name, customer.last_name, SUM(payment.amount) AS total_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name;
"""
print(pd.read_sql_query(query, conn))

    first_name last_name  total_payment
0        AARON     SELBY         110.76
1         ADAM     GOOCH         101.78
2       ADRIAN     CLARY          74.81
3        AGNES    BISHOP          98.77
4         ALAN      KAHN         124.74
..         ...       ...            ...
594     WILLIE   MARKHAM         101.75
595      WILMA  RICHARDS          91.80
596    YOLANDA    WEAVER         110.73
597     YVONNE   WATKINS          92.79
598    ZACHARY      HITE         146.69

[599 rows x 3 columns]


55-)Hangi filmler en uzun süre kiralanmış

In [55]:
query = """
SELECT film.title, MAX(julianday(rental.return_date) - julianday(rental.rental_date)) AS max_rental_days
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY max_rental_days DESC, title ASC;
"""
print(pd.read_sql_query(query, conn))

                  title  max_rental_days
0    HOLOCAUST HIGHBALL         9.249306
1       HIGHBALL POTTER         9.249306
2     CONEHEADS SMOOCHY         9.248611
3            PANIC CLUB         9.248611
4          TRAMP OTHERS         9.248611
..                  ...              ...
953          SQUAD FISH         5.908333
954   CONSPIRACY SPIRIT         5.809722
955         GLORY TRACY         5.120833
956      TRAFFIC HOBBIT         4.977778
957   DUFFEL APOCALYPSE         4.137500

[958 rows x 2 columns]


56-)Hangi filmler en kısa süre kiralanmış

In [56]:
query = """
SELECT film.title, MIN(julianday(rental.return_date) - julianday(rental.rental_date)) AS min_rental_days
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY min_rental_days ASC;
"""
print(pd.read_sql_query(query, conn))

                 title  min_rental_days
0    CURTAIN VIDEOTAPE         0.750000
1      DISCIPLE MOTHER         0.750000
2         ROAD ROXANNE         0.750694
3       HEAVEN FREEDOM         0.751389
4        KISSING DOLLS         0.751389
..                 ...              ...
953        MIXED DOORS         3.240972
954         EARLY HOME         3.807639
955    PANKY SUBMARINE         3.827083
956         WAGON JAWS         3.859028
957       ZHIVAGO CORE         3.875000

[958 rows x 2 columns]


57-)Müşterilerin kiraladığı filmler için ortalama ödeme mıktarını bulun

In [57]:
query = """
SELECT customer.first_name, customer.last_name, AVG(payment.amount) AS avg_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name;
"""
print(pd.read_sql_query(query, conn))

    first_name last_name  avg_payment
0        AARON     SELBY     4.615000
1         ADAM     GOOCH     4.626364
2       ADRIAN     CLARY     3.937368
3        AGNES    BISHOP     4.294348
4         ALAN      KAHN     4.797692
..         ...       ...          ...
594     WILLIE   MARKHAM     4.070000
595      WILMA  RICHARDS     4.590000
596    YOLANDA    WEAVER     4.101111
597     YVONNE   WATKINS     4.418571
598    ZACHARY      HITE     4.731935

[599 rows x 3 columns]


58-) En çok kiralanan filmlerin ortalama kiralama süresi nedir?

In [107]:
query = """
SELECT film.title, AVG(julianday(rental.return_date) - julianday(rental.rental_date)) AS avg_rental_days
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY COUNT(rental.rental_id) DESC, film.title ASC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

                 title  avg_rental_days
0   BUCKET BROTHERHOOD         4.948448
1     ROCKETEER MOTHER         5.386069
2       FORWARD TEMPLE         5.595247
3       GRIT CLOCKWORK         5.340864
4       JUGGLER HARDLY         5.509901
5  RIDGEMONT SUBMARINE         6.010909
6        SCALAWAG DUCK         4.069640
7        APACHE DIVINE         4.269064
8    GOODFELLAS SALUTE         5.219848
9         HOBBIT ALIEN         5.716711


59-)En az kiralanan filmlerin ortalama kira süresi nedir?

In [108]:
query = """
SELECT film.title, AVG(julianday(rental.return_date) - julianday(rental.rental_date)) AS avg_rental_days
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY COUNT(rental.rental_id) ASC, film.title ASC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

               title  avg_rental_days
0     HARDLY ROBBERS         6.948437
1        MIXED DOORS         5.953299
2        TRAIN BUNCH         3.349306
3   BRAVEHEART HUMAN         4.019861
4        BUNCH MINDS         5.524444
5  CONSPIRACY SPIRIT         4.010000
6       FEVER EMPIRE         5.755694
7  FREEDOM CLEOPATRA         2.959722
8    FULL FLATLINERS         3.767361
9        GLORY TRACY         3.678472


60-) En çok kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?

In [60]:
query = """
SELECT customer.first_name, customer.last_name, AVG(payment.amount) AS avg_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY SUM(payment.amount) DESC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  avg_payment
0       KARL      SEAL     4.923333
1    ELEANOR      HUNT     4.707391
2      CLARA      SHAW     4.656667
3     MARION    SNYDER     4.990000
4     RHONDA   KENNEDY     4.990000
5      TOMMY   COLLAZO     4.911053
6     WESLEY      BULL     4.440000
7        TIM      CARY     4.502821
8     MARCIA      DEAN     4.180476
9        ANA   BRADLEY     5.137059


61-) En az kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir

In [61]:
query = """
SELECT customer.first_name, customer.last_name, AVG(payment.amount) AS avg_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY SUM(payment.amount) ASC, customer.last_name ASC, customer.first_name ASC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  avg_payment
0   CAROLINE    BOWMAN     3.390000
1      LEONA    OBRIEN     3.632857
2      BRIAN     WYMAN     4.406667
3     JOHNNY    TURPIN     3.042632
4      ANNIE   RUSSELL     3.267778
5  KATHERINE    RIVERA     4.204286
6    TIFFANY    JORDAN     4.275714
7      ANITA   MORALES     4.190000
8     MATTIE   HOFFMAN     2.944545
9       KIRK   STCLAIR     3.411053


62-)Mağazalardaki toplam kiralama süresini bulunuz

In [62]:
query = """
SELECT store.store_id, SUM(julianday(rental.return_date) - julianday(rental.rental_date)) AS total_rental_days
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY store.store_id
ORDER BY store.store_id;
"""
print(pd.read_sql_query(query, conn))

   store_id  total_rental_days
0         1       39878.755556
1         2       39828.009028


63-)En uzun süre kiralanan film hangi mağazada kiralanmış

In [110]:
query = """
SELECT store.store_id, film.title, 
ROUND(MAX(julianday(rental.return_date) - julianday(rental.rental_date)),6) AS max_rental_days
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
GROUP BY store.store_id, film.title
ORDER BY max_rental_days DESC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id            title  max_rental_days
0         2  HIGHBALL POTTER         9.249306


64-) En kısa süre kiralanan film hangi mağazada kiralanmış

In [64]:
query = """
SELECT store.store_id, film.title, MIN(julianday(rental.return_date) - julianday(rental.rental_date)) AS min_rental_days
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
GROUP BY store.store_id, film.title
ORDER BY min_rental_days ASC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id              title  min_rental_days
0         2  CURTAIN VIDEOTAPE             0.75


65-) Her filmin ortalama kiralama süresi nedir?

In [65]:
query = """
SELECT film.title, AVG(julianday(rental.return_date) - julianday(rental.rental_date)) AS avg_rental_days
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY film.title;
"""
print(pd.read_sql_query(query, conn))

                 title  avg_rental_days
0     ACADEMY DINOSAUR         4.997128
1       ACE GOLDFINGER         5.642708
2     ADAPTATION HOLES         3.465625
3     AFFAIR PREJUDICE         4.758333
4          AFRICAN EGG         7.106629
..                 ...              ...
953     YOUNG LANGUAGE         4.624405
954         YOUTH KICK         5.562963
955       ZHIVAGO CORE         5.950087
956  ZOOLANDER FICTION         5.542647
957          ZORRO ARK         4.539180

[958 rows x 2 columns]


66-) Hangi filmler en çok kiralanan kategoride

In [66]:
query = """
SELECT film.title, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM film
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE category.name = (
    SELECT category.name
    FROM category
    JOIN film_category ON category.category_id = film_category.category_id
    JOIN film ON film_category.film_id = film.film_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY category.name
    ORDER BY COUNT(rental.rental_id) DESC, category.name ASC
    LIMIT 1
)
GROUP BY film.title, genre
ORDER BY rental_count DESC, film.title ASC;
"""
print(pd.read_sql_query(query, conn))

                  title   genre  rental_count
0   GLEAMING JAWBREAKER  Sports            29
1     TALENTED HOMICIDE  Sports            29
2        ROSES TREASURE  Sports            28
3        SATURDAY LAMBS  Sports            28
4   CHANCE RESURRECTION  Sports            27
..                  ...     ...           ...
68          SENSE GREEK  Sports             7
69           SQUAD FISH  Sports             7
70        BUBBLE GROSSE  Sports             6
71   GRACELAND DYNAMITE  Sports             6
72   MUSSOLINI SPOILERS  Sports             5

[73 rows x 3 columns]


67-)Hangi filmler en az kiralanan kategoride

In [67]:
query = """
SELECT film.title, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM film
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE category.name = (
    SELECT category.name
    FROM category
    JOIN film_category ON category.category_id = film_category.category_id
    JOIN film ON film_category.film_id = film.film_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    GROUP BY category.name
    ORDER BY COUNT(rental.rental_id) ASC, category.name ASC
    LIMIT 1
)
GROUP BY film.title, genre
ORDER BY rental_count ASC, film.title ASC;
"""
print(pd.read_sql_query(query, conn))

                     title  genre  rental_count
0          TERMINATOR CLUB  Music             6
1               YOUTH KICK  Music             6
2               CLUE GRAIL  Music             7
3               ELF MURDER  Music             7
4             HEAVENLY GUN  Music             7
5         GROSSE WONDERFUL  Music             8
6            REBEL AIRPORT  Music             8
7           RUNNER MADIGAN  Music             8
8          BIRDCAGE CASPER  Music             9
9           IMPACT ALADDIN  Music             9
10             LEGEND JEDI  Music             9
11              REDS POCUS  Music             9
12               WON DARES  Music             9
13     AMELIE HELLFIGHTERS  Music            10
14     JAWBREAKER BROOKLYN  Music            10
15           OLEANDER CLUE  Music            10
16         BIRCH ANTITRUST  Music            11
17             SONG HEDWIG  Music            11
18           MINORITY KISS  Music            12
19         CHAMBER ITALIAN  Music       

68-) Hangi mağazalarda en çok film kiralanmış

In [68]:
query = """
SELECT store.store_id, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY store.store_id
ORDER BY rental_count DESC, store.store_id ASC;
"""
print(pd.read_sql_query(query, conn))

   store_id  rental_count
0         1          8040
1         2          8004


69-) Hangi mağazalarda en az film kiralanmış

In [69]:
query = """
SELECT store.store_id, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY store.store_id
ORDER BY rental_count ASC, store.store_id ASC;
"""
print(pd.read_sql_query(query, conn))

   store_id  rental_count
0         2          8004
1         1          8040


70-) Hangi aktörler en çok filmde oynamış

In [70]:
query = """
SELECT actor.first_name, actor.last_name, COUNT(film_actor.film_id) AS film_count
FROM actor
JOIN film_actor ON actor.actor_id = film_actor.actor_id
GROUP BY actor.first_name, actor.last_name
ORDER BY film_count DESC, actor.last_name ASC, actor.first_name ASC;
"""
print(pd.read_sql_query(query, conn))

    first_name  last_name  film_count
0        SUSAN      DAVIS          54
1         GINA  DEGENERES          42
2       WALTER       TORN          41
3         MARY     KEITEL          40
4      MATTHEW     CARREY          39
..         ...        ...         ...
194      SISSY   SOBIESKI          18
195      JULIA  ZELLWEGER          16
196       JUDY       DEAN          15
197      JULIA    FAWCETT          15
198      EMILY        DEE          14

[199 rows x 3 columns]


71-) Hangi aktörler en az filmde rol almış

In [71]:
query = """
SELECT actor.first_name, actor.last_name, COUNT(film_actor.film_id) AS film_count
FROM actor
JOIN film_actor ON actor.actor_id = film_actor.actor_id
GROUP BY actor.first_name, actor.last_name
ORDER BY film_count ASC, actor.last_name ASC, actor.first_name ASC;
"""
print(pd.read_sql_query(query, conn))

    first_name  last_name  film_count
0        EMILY        DEE          14
1         JUDY       DEAN          15
2        JULIA    FAWCETT          15
3        JULIA  ZELLWEGER          16
4         ADAM      GRANT          18
..         ...        ...         ...
194    MATTHEW     CARREY          39
195       MARY     KEITEL          40
196     WALTER       TORN          41
197       GINA  DEGENERES          42
198      SUSAN      DAVIS          54

[199 rows x 3 columns]


72-) Hangi aktörlerin oynadığı filmler en çok kiralanmış

In [72]:
query = """
SELECT actor.first_name, actor.last_name, COUNT(rental.rental_id) AS rental_count
FROM actor
JOIN film_actor ON actor.actor_id = film_actor.actor_id
JOIN film ON film_actor.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY actor.first_name, actor.last_name
ORDER BY rental_count DESC, actor.last_name ASC, actor.first_name ASC;
"""
print(pd.read_sql_query(query, conn))

    first_name    last_name  rental_count
0        SUSAN        DAVIS           825
1         GINA    DEGENERES           753
2      MATTHEW       CARREY           678
3         MARY       KEITEL           674
4       ANGELA  WITHERSPOON           654
..         ...          ...           ...
194       JUDY         DEAN           255
195      JULIA      FAWCETT           255
196      SISSY     SOBIESKI           235
197      JULIA    ZELLWEGER           221
198      EMILY          DEE           216

[199 rows x 3 columns]


73-) Hangi aktörlerin oynadığı filmler en az kiralanmış

In [73]:
query = """
SELECT actor.first_name, actor.last_name, COUNT(rental.rental_id) AS rental_count
FROM actor
JOIN film_actor ON actor.actor_id = film_actor.actor_id
JOIN film ON film_actor.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY actor.first_name, actor.last_name
ORDER BY rental_count ASC, actor.last_name ASC, actor.first_name ASC;
"""
print(pd.read_sql_query(query, conn))

    first_name    last_name  rental_count
0        EMILY          DEE           216
1        JULIA    ZELLWEGER           221
2        SISSY     SOBIESKI           235
3         JUDY         DEAN           255
4        JULIA      FAWCETT           255
..         ...          ...           ...
194     ANGELA  WITHERSPOON           654
195       MARY       KEITEL           674
196    MATTHEW       CARREY           678
197       GINA    DEGENERES           753
198      SUSAN        DAVIS           825

[199 rows x 3 columns]


74-)Hangi kategorilerde en fazla aktör oynamış

In [74]:
query = """
SELECT category.name AS genre, COUNT(DISTINCT film_actor.actor_id) AS actor_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN film_actor ON film.film_id = film_actor.film_id
GROUP BY genre
ORDER BY actor_count DESC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  actor_count
0        Sports          182
1       Foreign          175
2           New          169
3   Documentary          168
4        Sci-Fi          167
5        Action          166
6     Animation          166
7        Travel          166
8        Family          164
9      Children          163
10     Classics          162
11        Drama          162
12       Horror          156
13        Games          150
14       Comedy          147
15        Music          144


75-)Hangi kategorilerde en az aktör oynamış

In [75]:
query = """
SELECT category.name AS genre, COUNT(DISTINCT film_actor.actor_id) AS actor_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN film_actor ON film.film_id = film_actor.film_id
GROUP BY genre
ORDER BY actor_count ASC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  actor_count
0         Music          144
1        Comedy          147
2         Games          150
3        Horror          156
4      Classics          162
5         Drama          162
6      Children          163
7        Family          164
8        Action          166
9     Animation          166
10       Travel          166
11       Sci-Fi          167
12  Documentary          168
13          New          169
14      Foreign          175
15       Sports          182


76-) Hangi kategorilerde oynayan filmler en çok kiralanmış

In [76]:
query = """
SELECT category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY genre
ORDER BY rental_count DESC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  rental_count
0        Sports          1179
1     Animation          1166
2        Action          1112
3        Sci-Fi          1101
4        Family          1096
5         Drama          1060
6   Documentary          1050
7       Foreign          1033
8         Games           969
9      Children           945
10       Comedy           941
11          New           940
12     Classics           939
13       Horror           846
14       Travel           837
15        Music           830


77-) Hangi kategorilerde oynayan filmler en az kiralanmış

In [77]:
query = """
SELECT category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY genre
ORDER BY rental_count ASC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  rental_count
0         Music           830
1        Travel           837
2        Horror           846
3      Classics           939
4           New           940
5        Comedy           941
6      Children           945
7         Games           969
8       Foreign          1033
9   Documentary          1050
10        Drama          1060
11       Family          1096
12       Sci-Fi          1101
13       Action          1112
14    Animation          1166
15       Sports          1179


78-)En fazla kazanç sağlayan kategoriler hangileridir

In [78]:
query = """
SELECT category.name AS genre, SUM(payment.amount) AS total_revenue
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY genre
ORDER BY total_revenue DESC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  total_revenue
0        Sports        5314.21
1        Sci-Fi        4756.98
2     Animation        4656.30
3         Drama        4587.39
4        Comedy        4383.58
5        Action        4375.85
6           New        4351.62
7         Games        4281.33
8       Foreign        4270.67
9        Family        4226.07
10  Documentary        4217.52
11       Horror        3722.54
12     Children        3655.55
13     Classics        3639.59
14       Travel        3549.64
15        Music        3417.72


79-) En az kazanç sağlayan kategoriler hangileri

In [79]:
query = """
SELECT category.name AS genre, SUM(payment.amount) AS total_revenue
FROM category
JOIN film_category ON category.category_id = film_category.category_id
JOIN film ON film_category.film_id = film.film_id
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY genre
ORDER BY total_revenue ASC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre  total_revenue
0         Music        3417.72
1        Travel        3549.64
2      Classics        3639.59
3      Children        3655.55
4        Horror        3722.54
5   Documentary        4217.52
6        Family        4226.07
7       Foreign        4270.67
8         Games        4281.33
9           New        4351.62
10       Action        4375.85
11       Comedy        4383.58
12        Drama        4587.39
13    Animation        4656.30
14       Sci-Fi        4756.98
15       Sports        5314.21


80-) En çok film kiralayan müşterilerin ortalama ödeme miktarı

In [80]:
query = """
SELECT customer.first_name, customer.last_name, AVG(payment.amount) AS avg_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY COUNT(rental.rental_id) DESC, customer.last_name ASC, customer.first_name ASC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  avg_payment
0    ELEANOR      HUNT     4.707391
1       KARL      SEAL     4.923333
2     MARCIA      DEAN     4.180476
3      CLARA      SHAW     4.656667
4      TAMMY   SANDERS     3.794878
5     WESLEY      BULL     4.440000
6        SUE    PETERS     3.865000
7        TIM      CARY     4.502821
8     RHONDA   KENNEDY     4.990000
9     MARION    SNYDER     4.990000


81-) En az film kiralayan müşterilerin ortalama ödeme miktarı

In [81]:
query = """
SELECT customer.first_name, customer.last_name, AVG(payment.amount) AS avg_payment
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY COUNT(rental.rental_id) ASC, customer.last_name ASC, customer.first_name ASC
LIMIT 10;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  avg_payment
0      BRIAN     WYMAN     4.406667
1    TIFFANY    JORDAN     4.275714
2      LEONA    OBRIEN     3.632857
3  KATHERINE    RIVERA     4.204286
4   CAROLINE    BOWMAN     3.390000
5      ANITA   MORALES     4.190000
6      JOANN   GARDNER     4.177500
7     JEROME    KENYON     4.615000
8     LESTER     KRAUS     4.115000
9    ANTONIO      MEEK     4.927500


82-) Hangi mağazada hangi kategoriler en çok kiralanmış

In [82]:
query = """
SELECT store.store_id, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
GROUP BY store.store_id, genre
ORDER BY rental_count DESC, store.store_id ASC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

    store_id        genre  rental_count
0          2       Sports           614
1          2    Animation           584
2          1    Animation           582
3          1       Family           565
4          1       Sports           565
5          1       Action           562
6          1       Sci-Fi           553
7          2       Action           550
8          2       Sci-Fi           548
9          2      Foreign           543
10         2        Drama           532
11         2       Family           531
12         1        Drama           528
13         1  Documentary           525
14         2  Documentary           525
15         1        Games           494
16         1      Foreign           490
17         1          New           489
18         1     Children           483
19         1     Classics           475
20         1       Comedy           475
21         2        Games           475
22         2       Comedy           466
23         2     Classics           464


83-) Hangi mağazada hangi kategoriler en az kiralanmış

In [83]:
query = """
SELECT store.store_id, category.name AS genre, COUNT(rental.rental_id) AS rental_count
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
GROUP BY store.store_id, genre
ORDER BY rental_count ASC, store.store_id ASC, genre ASC;
"""
print(pd.read_sql_query(query, conn))

    store_id        genre  rental_count
0          2        Music           398
1          1       Travel           399
2          1       Horror           423
3          2       Horror           423
4          1        Music           432
5          2       Travel           438
6          2          New           451
7          2     Children           462
8          2     Classics           464
9          2       Comedy           466
10         1     Classics           475
11         1       Comedy           475
12         2        Games           475
13         1     Children           483
14         1          New           489
15         1      Foreign           490
16         1        Games           494
17         1  Documentary           525
18         2  Documentary           525
19         1        Drama           528
20         2       Family           531
21         2        Drama           532
22         2      Foreign           543
23         2       Sci-Fi           548


84-) En fazla sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [84]:
query = """
SELECT store.store_id, COUNT(rental.rental_id) AS total_rentals
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY store.store_id
ORDER BY total_rentals DESC, store.store_id ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id  total_rentals
0         1           8040


85-)En az sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [85]:
query = """
SELECT store.store_id, COUNT(rental.rental_id) AS total_rentals
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
GROUP BY store.store_id
ORDER BY total_rentals ASC, store.store_id ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id  total_rentals
0         2           8004


86-)En fazla sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir

In [86]:
query = """
SELECT customer.first_name, customer.last_name, COUNT(rental.rental_id) AS total_rentals
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY customer.first_name, customer.last_name
ORDER BY total_rentals DESC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  total_rentals
0    ELEANOR      HUNT             46


87-) En az sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir?

In [87]:
query = """
SELECT customer.first_name, customer.last_name, COUNT(rental.rental_id) AS total_rentals
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY customer.first_name, customer.last_name
ORDER BY total_rentals ASC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  total_rentals
0      BRIAN     WYMAN             12


88-) En fazla sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir

In [88]:
query = """
SELECT film.title, COUNT(rental.rental_id) AS total_rentals
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY total_rentals DESC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

                title  total_rentals
0  BUCKET BROTHERHOOD             34


89-) En az sayıda kiiralanan film hangisidir ve toplam kiralama sayısı nedir

In [89]:
query = """
SELECT film.title, COUNT(rental.rental_id) AS total_rentals
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY total_rentals ASC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

            title  total_rentals
0  HARDLY ROBBERS              4


90-)En fazla kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir

In [90]:
query = """
SELECT customer.first_name, customer.last_name, SUM(payment.amount) AS total_revenue
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY total_revenue DESC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  total_revenue
0       KARL      SEAL         221.55


91-)En az kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir

In [91]:
query = """
SELECT customer.first_name, customer.last_name, SUM(payment.amount) AS total_revenue
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY customer.first_name, customer.last_name
ORDER BY total_revenue ASC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  total_revenue
0   CAROLINE    BOWMAN          50.85


92-)Hangi film hangi kategoride en çok kazanç sağlamıştır

In [92]:
query = """
SELECT genre, title, total_revenue
FROM (
    SELECT category.name AS genre, film.title AS title, SUM(payment.amount) AS total_revenue,
           ROW_NUMBER() OVER (
               PARTITION BY category.name
               ORDER BY SUM(payment.amount) DESC, film.title ASC
           ) AS rn
    FROM film
    JOIN film_category ON film.film_id = film_category.film_id
    JOIN category ON film_category.category_id = category.category_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY genre, film.title
)
WHERE rn = 1
ORDER BY genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre                title  total_revenue
0        Action     FOOL MOCKINGBIRD         175.77
1     Animation         DOGMA FAMILY         178.70
2      Children  BACKLASH UNDEFEATED         158.81
3      Classics          STEEL SANTA         141.77
4        Comedy            ZORRO ARK         214.69
5   Documentary            WIFE TURN         223.69
6         Drama         TORQUE BOUND         198.72
7        Family     RANGE MOONWALKER         179.73
8       Foreign       INNOCENT USUAL         191.74
9         Games       MASSACRE USUAL         179.70
10       Horror           LOLA AGENT         159.76
11        Music     TELEGRAPH VOYAGE         231.73
12          New          MAIDEN HOME         163.76
13       Sci-Fi    GOODFELLAS SALUTE         209.69
14       Sports       SATURDAY LAMBS         204.72
15       Travel   BUCKET BROTHERHOOD         180.66


93-)Hangi film hangi kategoride en az kazanç sağlamıştır

In [93]:
query = """
SELECT genre, title, total_revenue
FROM (
    SELECT category.name AS genre, film.title AS title, SUM(payment.amount) AS total_revenue,
           ROW_NUMBER() OVER (
               PARTITION BY category.name
               ORDER BY SUM(payment.amount) ASC, film.title ASC
           ) AS rn
    FROM film
    JOIN film_category ON film.film_id = film_category.film_id
    JOIN category ON film_category.category_id = category.category_id
    JOIN inventory ON film.film_id = inventory.film_id
    JOIN rental ON inventory.inventory_id = rental.inventory_id
    JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY genre, film.title
)
WHERE rn = 1
ORDER BY genre ASC;
"""
print(pd.read_sql_query(query, conn))

          genre                 title  total_revenue
0        Action     MONTEZUMA COMMAND          11.91
1     Animation      GHOSTBUSTERS ELF          11.91
2      Children     COMANCHEROS ENEMY           9.94
3      Classics    CRUELTY UNFORGIVEN           7.93
4        Comedy     FREEDOM CLEOPATRA           5.95
5   Documentary     DUFFEL APOCALYPSE           6.93
6         Drama         HAROLD FRENCH           9.92
7        Family           SOUP WISDOM          12.92
8       Foreign  BALLROOM MOCKINGBIRD          10.92
9         Games           GLORY TRACY          14.95
10       Horror           TEXAS WATCH           5.94
11        Music         REBEL AIRPORT           7.92
12          New      OKLAHOMA JUMANJI           5.94
13       Sci-Fi     STALLION SUNDANCE           8.93
14       Sports       VICTORY ACADEMY          12.91
15       Travel   VALENTINE VANISHING          16.88


94-)En fazla kazanç sağlayan mağaza hangisidir, toplam kazancı nedir 

In [94]:
query = """
SELECT store.store_id, SUM(payment.amount) AS total_revenue
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY store.store_id
ORDER BY total_revenue DESC, store.store_id ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id  total_revenue
0         2       33881.94


95-)En az kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?

In [95]:
query = """
SELECT store.store_id, SUM(payment.amount) AS total_revenue
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY store.store_id
ORDER BY total_revenue ASC, store.store_id ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id  total_revenue
0         1       33524.62


96-)En fazla sayıda farklı film kiralayan müşteri hangisidir

In [96]:
query = """
SELECT customer.first_name, customer.last_name, COUNT(DISTINCT rental.inventory_id) AS unique_rentals
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY customer.first_name, customer.last_name
ORDER BY unique_rentals DESC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  unique_rentals
0    ELEANOR      HUNT              46


97-) En az sayıda farklı film kiralayan müşteri hangsidir?

In [97]:
query = """
SELECT customer.first_name, customer.last_name, COUNT(DISTINCT rental.inventory_id) AS unique_rentals
FROM customer
JOIN rental ON customer.customer_id = rental.customer_id
GROUP BY customer.first_name, customer.last_name
ORDER BY unique_rentals ASC, customer.last_name ASC, customer.first_name ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

  first_name last_name  unique_rentals
0      BRIAN     WYMAN              12


98-)En fazla sayıda farklı müşteri tarafından kiralanan film hangisidir?

In [98]:
query = """
SELECT film.title, COUNT(DISTINCT rental.customer_id) AS unique_customers
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY unique_customers DESC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

                title  unique_customers
0  BUCKET BROTHERHOOD                33


99-)En az sayıda farklı müşteri tarafından kiralanan film hangisidir

In [99]:
query = """
SELECT film.title, COUNT(DISTINCT rental.customer_id) AS unique_customers
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
GROUP BY film.title
ORDER BY unique_customers ASC, film.title ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

            title  unique_customers
0  HARDLY ROBBERS                 4


100-)Hangi mağazada hangi kategoride en fazla kazanç sağlanmıştır

In [100]:
query = """
SELECT store.store_id, category.name AS genre, SUM(payment.amount) AS total_revenue
FROM store
JOIN staff ON store.store_id = staff.store_id
JOIN rental ON staff.staff_id = rental.staff_id
JOIN inventory ON rental.inventory_id = inventory.inventory_id
JOIN film ON inventory.film_id = film.film_id
JOIN film_category ON film.film_id = film_category.film_id
JOIN category ON film_category.category_id = category.category_id
JOIN payment ON rental.rental_id = payment.rental_id
GROUP BY store.store_id, genre
ORDER BY total_revenue DESC, store.store_id ASC, genre ASC
LIMIT 1;
"""
print(pd.read_sql_query(query, conn))

   store_id   genre  total_revenue
0         2  Sports        2761.85


## GÖREV SONU